<a href="https://colab.research.google.com/github/zinduaschool/ml-notes/blob/main/25h-notes/Week%204%20-%20Deep%20Learning%20Foundations/Deep_Learning_Practice_Day_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### We will build a simple multi layer perceptron to predict hourly wage for employees and compare this with a linear regression model.

In [41]:
## Import the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.preprocessing import StandardScaler #important to scale your data before training a nn.
from sklearn.linear_model import LinearRegression
from keras.optimizers import SGD
from keras.callbacks import EarlyStopping
from keras.layers import Dropout

In [42]:
# load my data
df=pd.read_csv('https://assets.datacamp.com/production/repositories/654/datasets/8a57adcdb5bfb3e603dad7d3c61682dfe63082b8/hourly_wages.csv')
df.head()

,wage_per_hour,union,education_yrs,experience_yrs,age,female,marr,south,manufacturing,construction
0,5.10,0,8,21,35,1,1,0,1,0
1,4.95,0,9,42,57,1,1,0,1,0
2,6.67,0,12,1,19,0,0,0,1,0
3,4.00,0,12,4,22,0,0,0,0,0
4,7.50,0,12,17,35,0,1,0,0,0


In [43]:
# so we want to predict wage per hour based on the attributes of the worker
df.describe()

,wage_per_hour,union,education_yrs,experience_yrs,age,female,marr,south,manufacturing,construction
count,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000,534.000000
mean,9.024064,0.179775,13.018727,17.822097,36.833333,0.458801,0.655431,0.292135,0.185393,0.044944
std,5.139097,0.384360,2.615373,12.379710,11.726573,0.498767,0.475673,0.455170,0.388981,0.207375
min,1.000000,0.000000,2.000000,0.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.250000,0.000000,12.000000,8.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.780000,0.000000,12.000000,15.000000,35.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,11.250000,0.000000,15.000000,26.000000,44.000000,1.000000,1.000000,1.000000,0.000000,0.000000
max,44.500000,1.000000,18.000000,55.000000,64.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [44]:
# This data does not require much processing
X=df.drop('wage_per_hour',axis=1).values
y=df['wage_per_hour'].values



In [45]:
# scale it
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

# train test split
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)

(427, 9)
(107, 9)


In [46]:
# Define my model
n_cols=X_train.shape[1]
model = Sequential()
model.add(Dense(100, activation='relu'))
model.add(Dense(50, activation='relu'))
model.add(Dense(20, activation='relu'))
model.add(Dense(1))

In [47]:
# compile our model
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train,y_train,validation_split=0.2,epochs=20,verbose=False)

In [48]:
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 100)            │         1,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 20)             │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,275 (83.11 KB)

 Trainable params: 7,091 (27.70 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 14,184 (55.41 KB)

In [49]:
# predict the test data
y_pred=model.predict(X_test)
mean_squared_error(y_test,y_pred)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


21.418271907099093

#### Comparing to a linear regression model

In [50]:
## Compare this with a linear regression model
lin_reg=LinearRegression()
lin_reg.fit(X_train,y_train)
y_pred_lin=lin_reg.predict(X_test)
mean_squared_error(y_test,y_pred_lin)

21.24712426009875

In [51]:
## lets optimize the nn to see if it can improve
model=Sequential()
model.add(Dense(100,activation="relu"))
model.add(Dense(50,activation="relu"))
model.add(Dense(20,activation="relu"))
model.add(Dense(1,activation='relu'))# since this is a regression problem the class of the output is just one

In [52]:
# compile the model with SGD
learning_rates = [0.001,0.0001,0.1,0.003,0.3,0.0004,0.000001,0.0005]
for lr in learning_rates:
    model=model
    sgd=SGD(learning_rate=lr)
    model.compile(optimizer=sgd,loss='mean_squared_error')
    model.fit(X_train,y_train)

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 107.3158  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 110.3681  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 70.9608  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 36.1320  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 56.9232  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 100.9650 
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 107.5851  
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 104.8462 


In [53]:
## implement drop out
model=Sequential()
model.add(Dense(100,activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(50,activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(20,activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(1,activation='relu'))

In [54]:
### stop overfitting by adding an early stopping criteria and drop out regularization
callbacks=EarlyStopping(patience=2)
sgd=SGD(learning_rate=0.001)
model.compile(optimizer=sgd,loss='mean_squared_error')
model.fit(X_train,y_train,epochs=30,validation_split=0.2,callbacks=[EarlyStopping(patience=2)])

Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 107.5223 - val_loss: 104.0250
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 91.5212 - val_loss: 90.2277
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 78.5262 - val_loss: 67.3340
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 65.1945 - val_loss: 39.3364
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 40.0204 - val_loss: 26.0599
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 27.1804 - val_loss: 23.5461
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 26.2051 - val_loss: 22.3410
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 26.6920 - val_loss: 21.0422
Epoch 9/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 28.6055 - val_loss: 20.9888
Epoch 10/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 30.2326 - val_loss: 20.1458
Epoch 11/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 30.2224 - val_loss: 20.2863
Epoch 12/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/st